In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import itertools
import random
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

display(HTML("<style>.container { width:100% !important; }</style>"))
plt.rcParams['figure.figsize'] = (24, 8)
sns.set_theme(style="darkgrid")

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'


In [2]:
# 2. Data Loading (Fast Subset for Active Miner)
def load_all_data():
    all_data = []
    files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
    random.seed(42)
    random.shuffle(files)
    
    for file_path in files:
        if len(all_data) >= 50: # 50 stocks for fast grid searching
            break
        symbol = os.path.basename(file_path).replace('.csv', '')
        try:
            df = pd.read_csv(file_path)
            df = df.dropna(subset=['Close'])
            
            # Price Filter
            if len(df) > 0:
                latest_price = df['Close'].iloc[-1]
                if latest_price < 90 or latest_price > 600:
                    continue
                    
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.sort_values('Date').reset_index(drop=True)
            
            # Anomaly Filter
            df['Daily_Ret'] = df['Close'].pct_change()
            df = df[(df['Daily_Ret'].isna()) | (df['Daily_Ret'].abs() <= 0.20)]
            
            df['Symbol'] = symbol
            all_data.append(df)
        except Exception as e:
            pass
            
    print(f"Loaded {len(all_data)} stocks.")
    return pd.concat(all_data, ignore_index=True)

data = load_all_data()
num_stocks = data['Symbol'].nunique()
print(f"Unique Stocks: {num_stocks}")


Loaded 50 stocks.
Unique Stocks: 50


In [3]:
# 3. Fast Feature Computation (Vectorized)
LOOKBACKS = [8, 13, 21]
MAX_HOLD = 13

def enrich_data(df):
    df = df.copy()
    
    # Calculate all Forward Returns at once
    for h in [5, 8, 13]:
        df[f'Fwd_{h}d'] = df['Close'].shift(-h) / df['Close'] - 1
        
    # Calculate Features for all lookbacks
    for p in LOOKBACKS:
        # ER
        change = abs(df['Close'] - df['Close'].shift(p))
        volatility = df['Close'].diff().abs().rolling(p).sum()
        df[f'ER_{p}'] = change / volatility
        
        # WMA Dist
        weights = np.arange(1, p + 1)
        wma = df['Close'].rolling(p).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
        df[f'WMA_Dist_{p}'] = (df['Close'] - wma) / wma
        
    return df

print("Pre-computing all lookbacks and holds...")
_data_list = []
for sym, grp in data.groupby('Symbol'):
    _data_list.append(enrich_data(grp))
data = pd.concat(_data_list, ignore_index=True)

# Use Unseen Data
test_data = data[data['Date'] >= '2023-01-01']

# Calculate exact years of test data
min_date = test_data['Date'].min()
max_date = test_data['Date'].max()
years = (max_date - min_date).days / 365.25
print(f"Unseen Data duration: {years:.2f} years")


Pre-computing all lookbacks and holds...


Unseen Data duration: 3.57 years


In [4]:
# 4. The High-Frequency Grid Search
er_thresholds = [0.3, 0.4, 0.5]
wma_thresholds = [0.05, 0.08, 0.10]
hold_days = [5, 8, 13]

results = []
total_combinations = len(LOOKBACKS) * len(er_thresholds) * len(wma_thresholds) * len(hold_days)
print(f"Running Active Miner Grid Search: {total_combinations} combinations...")

for l_idx, lookback in enumerate(LOOKBACKS):
    er_col = f'ER_{lookback}'
    wma_col = f'WMA_Dist_{lookback}'
    
    for er_t in er_thresholds:
        for wma_t in wma_thresholds:
            # Filter trades once per parameter set
            # Added a lower bound (-0.03) to prevent catching falling knives!
            trades = test_data[(test_data[er_col] > er_t) & (test_data[wma_col] < wma_t) & (test_data[wma_col] > -0.03)]
            num_trades = len(trades)
            
            # Calculate Frequency
            trades_per_stock_per_year = num_trades / num_stocks / years if num_stocks > 0 and years > 0 else 0
            
            for hold in hold_days:
                fwd_col = f'Fwd_{hold}d'
                valid_trades = trades.dropna(subset=[fwd_col])
                n_valid = len(valid_trades)
                
                if n_valid == 0:
                    continue
                    
                win_rate = (valid_trades[fwd_col] > 0).mean() * 100
                avg_ret = valid_trades[fwd_col].mean() * 100
                
                # Active Miner Fitness Constraint
                # We harshly penalize anything under 3 trades/stock/year, and reward high frequency and profit
                if trades_per_stock_per_year < 3:
                    fitness = 0 
                else:
                    # Fitness favors high win rates, high returns, and high frequency
                    fitness = (win_rate / 100) * avg_ret * trades_per_stock_per_year
                
                results.append({
                    'Lookback': lookback,
                    'ER_Entry': f"> {er_t}",
                    'WMA_Avoid': f"-0.03 to {wma_t}",
                    'Hold_Days': hold,
                    'Total_Trades': n_valid,
                    'Trades_Per_Stock_Per_Yr': trades_per_stock_per_year,
                    'Test_WinRate': win_rate,
                    'Test_AvgReturn': avg_ret,
                    'Fitness': fitness
                })

results_df = pd.DataFrame(results)
# Sort by fitness, dropping the 0-fitness (low frequency) setups
valid_results = results_df[results_df['Fitness'] > 0].sort_values(by='Fitness', ascending=False).reset_index(drop=True)

print("\n=== ACTIVE MINER LEADERBOARD (Unseen 2023+ Data) ===")
if len(valid_results) > 0:
    display(valid_results.head(15))
else:
    print("NO rules generated more than 3 trades per stock per year with this grid.")


Running Active Miner Grid Search: 81 combinations...



=== ACTIVE MINER LEADERBOARD (Unseen 2023+ Data) ===


,Lookback,ER_Entry,WMA_Avoid,Hold_Days,Total_Trades,Trades_Per_Stock_Per_Yr,Test_WinRate,Test_AvgReturn,Fitness
0,8,> 0.3,-0.03 to 0.1,13,18518,105.426343,49.881197,0.735387,38.672459
1,8,> 0.3,-0.03 to 0.08,13,18310,104.254628,49.879847,0.715264,37.195215
2,13,> 0.3,-0.03 to 0.1,13,12635,71.984804,51.301939,0.941316,34.762433
3,8,> 0.3,-0.03 to 0.05,13,17412,99.102444,49.741558,0.670839,33.069073
4,13,> 0.3,-0.03 to 0.08,13,12280,69.955326,51.099349,0.901218,32.215589
5,8,> 0.4,-0.03 to 0.1,13,13787,78.364766,49.989120,0.748515,29.322229
6,8,> 0.4,-0.03 to 0.08,13,13602,77.321995,50.007352,0.726135,28.077239
7,13,> 0.3,-0.03 to 0.05,13,10948,62.386830,51.096091,0.833997,26.585527
8,8,> 0.4,-0.03 to 0.05,13,12808,72.775292,49.867270,0.676096,24.536235
9,13,> 0.4,-0.03 to 0.1,13,7814,44.429873,52.022012,1.042905,24.104986
